In [4]:
from __future__ import annotations
import pandas as pd
from pathlib import Path

DATE_COL = "Date"

def merge_latest_wins_iso(base_path: Path, new_path: Path, out_path: Path) -> pd.DataFrame:
    def load_as_str(path: Path) -> pd.DataFrame:
        df = pd.read_csv(path, dtype=str)
        df.columns = [c.strip() for c in df.columns]
        return df

    def align_to_schema(df_ref: pd.DataFrame, df_other: pd.DataFrame) -> pd.DataFrame:
        ref_cols = list(df_ref.columns)
        lower_map = {c.lower(): c for c in ref_cols}
        rename = {}
        for c in df_other.columns:
            if c in ref_cols:
                rename[c] = c
            elif c.lower() in lower_map:
                rename[c] = lower_map[c.lower()]
        df2 = df_other.rename(columns=rename)
        for c in ref_cols:
            if c not in df2.columns:
                df2[c] = pd.NA
        return df2[ref_cols]

    def parse_iso_date_col(df: pd.DataFrame, date_col: str) -> pd.DataFrame:
        s = df[date_col].astype(str).str.strip()
        dt = pd.to_datetime(s, errors="coerce", format="%Y-%m-%d")
        df = df.copy()
        df[date_col] = dt
        # Drop bad/blank dates
        df = df.dropna(subset=[date_col]).copy()
        return df

    def coerce_numeric(df: pd.DataFrame, date_col: str) -> pd.DataFrame:
        out = df.copy()
        for c in out.columns:
            if c == date_col:
                continue
            out[c] = out[c].map(lambda x: x.lstrip("'") if isinstance(x, str) else x)
            out[c] = out[c].replace({"": pd.NA, "NaN": pd.NA, "nan": pd.NA})
            out[c] = pd.to_numeric(out[c], errors="coerce")
        return out

    base = load_as_str(base_path)
    new  = load_as_str(new_path)
    new  = align_to_schema(base, new)

    base = parse_iso_date_col(base, DATE_COL)
    new  = parse_iso_date_col(new, DATE_COL)

    # Optional: coerce numerics
    base = coerce_numeric(base, DATE_COL)
    new  = coerce_numeric(new, DATE_COL)

    print(f"Base range: {base[DATE_COL].min().date()} → {base[DATE_COL].max().date()}  rows={len(base)}")
    print(f"New  range: {new[DATE_COL].min().date()} → {new[DATE_COL].max().date()}  rows={len(new)}")

    base["__src__"] = "base"
    new["__src__"]  = "new"
    merged = pd.concat([base, new], ignore_index=True)
    merged = merged.sort_values([DATE_COL, "__src__"]).drop_duplicates(subset=[DATE_COL], keep="last")
    merged = merged.sort_values(DATE_COL).reset_index(drop=True)

    # Guardrail: merged bounds must be within the union of inputs
    min_expected = min(base[DATE_COL].min(), new[DATE_COL].min())
    max_expected = max(base[DATE_COL].max(), new[DATE_COL].max())
    min_actual, max_actual = merged[DATE_COL].min(), merged[DATE_COL].max()
    assert min_actual >= min_expected and max_actual <= max_expected, (
        f"Merged dates out of expected bounds: {min_actual} .. {max_actual} "
        f"(expected within {min_expected} .. {max_expected})"
    )

    merged = merged.drop(columns="__src__", errors="ignore")
    merged.to_csv(out_path, index=False, date_format="%Y-%m-%d")
    print(f"Saved: {out_path.resolve()}")
    return merged

# Example usage:
merged_df = merge_latest_wins_iso(
    Path("historical_base.csv"),
    Path("historical_new.csv"),
    Path("historical_merged.csv"),
)
display(merged_df.tail())


Base range: 2024-06-01 → 2025-07-31  rows=426
New  range: 2025-07-01 → 2025-09-30  rows=92
Saved: C:\GitHubRepositories\leachit_ep\backend\public\historical_merged.csv


C:\Users\expg\AppData\Local\Temp\ipykernel_33204\2550639560.py:61: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  base["__src__"] = "base"
C:\Users\expg\AppData\Local\Temp\ipykernel_33204\2550639560.py:62: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  new["__src__"]  = "new"


,Date,Percent_Solids,WAD_CN_Tailings_ppm,Ph_Leach_Tank_01,Ph_Leach_Tank_02,Ph_Cil_Tank_01,Ph_Cil_Tank_02,Ph_Cil_Tank_03,Ph_Cil_Tank_04,Ph_Cil_Tank_05,...,Leach_Feed_Dry_t,Au_CIL_Solids_g,Au_Tailings_Solids_g,Au_Leach_Feed_g,Ag_Leach_Feed_g,Au_Tailings_g,Ag_Cil_Tailings_g,Au_Leach_Feed_Grade_gpt,Au_Cil_Tailings_Residuals_gpt,Recovery_pct
482,2025-09-26,51.541667,268.0,12.075,12.020,11.945,11.880,11.885,11.925,11.92,...,2402.5000,4228.400000,540.562500,4236.305720,11423.561435,547.338832,2951.797539,1.763291,0.225,87.215909
483,2025-09-27,51.458333,0.0,5.510,5.905,5.530,5.410,5.330,5.430,5.50,...,2586.8180,4798.547390,646.704500,4800.987587,12385.250916,652.804992,3265.653503,1.855943,0.250,86.522911
484,2025-09-28,51.083333,0.0,11.660,11.325,11.135,11.025,10.780,10.940,11.34,...,2589.4728,4997.682504,582.631380,5002.641788,12465.772189,587.590664,2660.883954,1.931915,0.225,88.341969
485,2025-09-29,51.173913,338.0,12.040,11.820,11.680,11.560,11.470,11.420,11.30,...,2458.3987,4498.869621,651.475656,4501.215230,10712.901166,659.685287,3380.831875,1.830954,0.265,85.519126
486,2025-09-30,51.083333,290.0,11.305,11.470,11.570,11.655,11.710,11.775,11.82,...,2597.5985,5415.992872,740.315573,5418.480296,12361.621023,749.021553,3616.999123,2.085958,0.285,86.330935
